In [182]:
from pathlib import Path

import warnings

import polars as pl

warnings.filterwarnings("ignore")

pl.Config.set_tbl_rows(10)
pl.Config.set_tbl_cols(20)

polars.config.Config

In [183]:
PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data"

RAW_DATA_DIR = DATA_DIR / "raw"

PRODUCTS_PATH = RAW_DATA_DIR / "Products.csv"
WAREHOUSES_PATH = RAW_DATA_DIR / "Warehouses.csv"
SALES_PATH = RAW_DATA_DIR / "Sales_History.csv"
INVENTORY_PATH = RAW_DATA_DIR / "Inventory.csv"

In [184]:
PROJECT_ROOT = Path.cwd().parent

RAW_DIR = PROJECT_ROOT / "data" / "raw"

INTERIM_DIR = PROJECT_ROOT / "data" / "interim"

INTERIM_DIR.mkdir(parents=True, exist_ok=True)

In [185]:
products_df = pl.read_csv(RAW_DIR / "Products.csv")

warehouses_df = pl.read_csv(RAW_DIR / "Warehouses.csv")

sales_df = pl.read_csv(RAW_DIR / "Sales_History.csv")

inventory_df = pl.read_csv(RAW_DIR / "Inventory.csv")

In [186]:
required_sales_columns = [
    "product_id",
    "hub_id",
    "year",
    "month",
]

missing = [
    col
    for col in required_sales_columns
    if col not in sales_df.columns
]

assert len(missing) == 0, f"Missing columns: {missing}"

print("Sales schema validated.")

Sales schema validated.


In [187]:
sales_df = sales_df.with_columns(
    [
        pl.col("year").cast(pl.Int16),
        pl.col("month").cast(pl.Int8),
        pl.col("promotion").cast(pl.Int8),
        pl.col("festival_flag").cast(pl.Int8),
    ]
)

In [188]:
merged_df = (
    sales_df
    .join(
        products_df,
        on="product_id",
        how="left",
    )
    .join(
        warehouses_df,
        on="hub_id",
        how="left",
    )
    .join(
        inventory_df,
        on=["hub_id", "product_id"],
        how="left",
    )
)

In [189]:
print(merged_df.shape)

display(merged_df.head())

display(merged_df.null_count())

(180000, 19)


year,month,hub_id,product_id,opening_stock,quantity_sold,promotion,discount_percentage,festival_flag,returns,closing_stock,product_name,sku,category,brand,hub_name,city,current_stock,last_updated
i16,i8,str,str,i64,i64,i8,i64,i8,i64,i64,str,str,str,str,str,str,i64,str
2023,1,"""HUB001""","""P000001""",527,132,0,0,1,6,401,"""Apple Fashion 1""","""SKU00001""","""Fashion""","""Apple""","""Chennai Hub""","""Chennai""",216,"""2025-12-31"""
2023,1,"""HUB001""","""P000002""",506,50,0,0,1,0,456,"""LG Sports 2""","""SKU00002""","""Sports""","""LG""","""Chennai Hub""","""Chennai""",808,"""2025-12-31"""
2023,1,"""HUB001""","""P000003""",427,111,0,0,1,2,318,"""Samsung Home 3""","""SKU00003""","""Home""","""Samsung""","""Chennai Hub""","""Chennai""",418,"""2025-12-31"""
2023,1,"""HUB001""","""P000004""",531,77,0,0,1,3,457,"""Puma Fashion 4""","""SKU00004""","""Fashion""","""Puma""","""Chennai Hub""","""Chennai""",431,"""2025-12-31"""
2023,1,"""HUB001""","""P000005""",855,28,0,0,1,1,828,"""Apple Electronics 5""","""SKU00005""","""Electronics""","""Apple""","""Chennai Hub""","""Chennai""",205,"""2025-12-31"""


year,month,hub_id,product_id,opening_stock,quantity_sold,promotion,discount_percentage,festival_flag,returns,closing_stock,product_name,sku,category,brand,hub_name,city,current_stock,last_updated
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [190]:
merged_df.write_parquet(
    INTERIM_DIR / "merged.parquet"
)

print("Merged dataset saved successfully.")

Merged dataset saved successfully.


In [191]:
products = products_df.clone()

warehouses = warehouses_df.clone()

sales = sales_df.clone()

inventory = inventory_df.clone()

In [192]:
sales = sales.with_columns(
    [
        pl.col("year").cast(pl.Int16),
        pl.col("month").cast(pl.Int8),

        pl.col("opening_stock").cast(pl.Int32),
        pl.col("closing_stock").cast(pl.Int32),

        pl.col("quantity_sold").cast(pl.Int32),

        pl.col("promotion").cast(pl.Int8),
        pl.col("festival_flag").cast(pl.Int8),

        pl.col("returns").cast(pl.Int32),

        pl.col("discount_percentage").cast(pl.Float32),
    ]
)

inventory = inventory.with_columns(
    [
        pl.col("current_stock").cast(pl.Int32),
    ]
)

In [193]:
products = products.unique()

warehouses = warehouses.unique()

sales = sales.unique()

inventory = inventory.unique()

In [194]:
sales = (
    sales
    .filter(pl.col("month").is_between(1, 12))
    .filter(pl.col("quantity_sold") >= 0)
    .filter(pl.col("opening_stock") >= 0)
    .filter(pl.col("closing_stock") >= 0)
    .filter(pl.col("returns") >= 0)
)

In [195]:
products = products.with_columns(
    [
        pl.col("product_id").str.strip_chars(),
    ]
)

warehouses = warehouses.with_columns(
    [
        pl.col("hub_id").str.strip_chars(),
    ]
)

sales = sales.with_columns(
    [
        pl.col("product_id").str.strip_chars(),
        pl.col("hub_id").str.strip_chars(),
    ]
)

inventory = inventory.with_columns(
    [
        pl.col("product_id").str.strip_chars(),
        pl.col("hub_id").str.strip_chars(),
    ]
)

In [196]:
merged = (
    sales
    .join(
        products,
        on="product_id",
        how="left",
    )
)

print(merged.shape)

(180000, 15)


In [197]:
merged = (
    merged
    .join(
        warehouses,
        on="hub_id",
        how="left",
    )
)

print(merged.shape)

(180000, 17)


In [198]:
merged.columns


['year',
 'month',
 'hub_id',
 'product_id',
 'opening_stock',
 'quantity_sold',
 'promotion',
 'discount_percentage',
 'festival_flag',
 'returns',
 'closing_stock',
 'product_name',
 'sku',
 'category',
 'brand',
 'hub_name',
 'city']

In [199]:
duplicate_keys = (
    merged
    .group_by(
        [
            "product_id",
            "hub_id",
            "year",
            "month",
        ]
    )
    .len()
    .filter(pl.col("len") > 1)
)

print(f"Duplicate Business Keys: {duplicate_keys.height}")

Duplicate Business Keys: 0


In [200]:
merged = merged.sort(
    ["product_id", "hub_id", "year", "month"]
)

In [201]:
merged.columns

['year',
 'month',
 'hub_id',
 'product_id',
 'opening_stock',
 'quantity_sold',
 'promotion',
 'discount_percentage',
 'festival_flag',
 'returns',
 'closing_stock',
 'product_name',
 'sku',
 'category',
 'brand',
 'hub_name',
 'city']

In [202]:
merged = merged.with_columns(
    [
        pl.col("quantity_sold")
        .shift(1)
        .over(["product_id", "hub_id"])
        .alias("sales_last_1"),

        pl.col("quantity_sold")
        .shift(2)
        .over(["product_id", "hub_id"])
        .alias("sales_last_2"),

        pl.col("quantity_sold")
        .shift(3)
        .over(["product_id", "hub_id"])
        .alias("sales_last_3"),
    ]
)

In [203]:
merged = merged.with_columns(
    pl.col("quantity_sold")
    .shift(-1)
    .over(["product_id", "hub_id"])
    .alias("target_next_month_demand")
)

In [204]:
merged = merged.with_columns(
    (
        (
            pl.col("sales_last_1")
            + pl.col("sales_last_2")
            + pl.col("sales_last_3")
        ) / 3
    ).alias("avg_sales_3")
)

In [205]:
merged.columns

['year',
 'month',
 'hub_id',
 'product_id',
 'opening_stock',
 'quantity_sold',
 'promotion',
 'discount_percentage',
 'festival_flag',
 'returns',
 'closing_stock',
 'product_name',
 'sku',
 'category',
 'brand',
 'hub_name',
 'city',
 'sales_last_1',
 'sales_last_2',
 'sales_last_3',
 'target_next_month_demand',
 'avg_sales_3']

In [206]:
merged = (
    merged
    .drop_nulls(
        [
            "sales_last_1",
            "sales_last_2",
            "sales_last_3",
            "target_next_month_demand",
        ]
    )
)

In [207]:
print(merged.columns)

['year', 'month', 'hub_id', 'product_id', 'opening_stock', 'quantity_sold', 'promotion', 'discount_percentage', 'festival_flag', 'returns', 'closing_stock', 'product_name', 'sku', 'category', 'brand', 'hub_name', 'city', 'sales_last_1', 'sales_last_2', 'sales_last_3', 'target_next_month_demand', 'avg_sales_3']


In [208]:
FEATURE_COLUMNS = [
    "year",
    "month",

    "product_id",
    "hub_id",

    "category",
    "brand",
    "city",
    "product_name",

    "opening_stock",

    "sales_last_1",
    "sales_last_2",
    "sales_last_3",
    "avg_sales_3",

    "promotion",
    "discount_percentage",
    "festival_flag",
    "returns",

    "target_next_month_demand",
]

training_df = merged.select(FEATURE_COLUMNS)

In [209]:
training_df.columns

['year',
 'month',
 'product_id',
 'hub_id',
 'category',
 'brand',
 'city',
 'product_name',
 'opening_stock',
 'sales_last_1',
 'sales_last_2',
 'sales_last_3',
 'avg_sales_3',
 'promotion',
 'discount_percentage',
 'festival_flag',
 'returns',
 'target_next_month_demand']

In [210]:
training_df = training_df.sort(
    ["year", "month"]
)

n = training_df.height

train_end = int(n * 0.70)
val_end = int(n * 0.85)

train_df = training_df.slice(0, train_end)

validation_df = training_df.slice(
    train_end,
    val_end - train_end,
)

test_df = training_df.slice(
    val_end,
    n - val_end,
)

In [211]:
TARGET = "target_next_month_demand"

FEATURES = [
    c for c in training_df.columns
    if c != TARGET
]

In [212]:
X_train = train_df.select(FEATURES).to_pandas()
X_val = validation_df.select(FEATURES).to_pandas()
X_test = test_df.select(FEATURES).to_pandas()

y_train = train_df[TARGET].to_numpy()
y_val = validation_df[TARGET].to_numpy()
y_test = test_df[TARGET].to_numpy()

In [213]:
CATEGORICAL_FEATURES = [
    "product_id",
    "hub_id",
    "category",
    "brand",
    "city",
    "product_name",
]

In [214]:
NUMERIC_FEATURES = [
    "year",
    "month",
    "opening_stock",
    "sales_last_1",
    "sales_last_2",
    "sales_last_3",
    "avg_sales_3",
    "promotion",
    "discount_percentage",
    "festival_flag",
    "returns",
]

In [232]:
X_train_cat = X_train.copy()
X_val_cat = X_val.copy()
X_test_cat = X_test.copy()

In [236]:
for col in CATEGORICAL_FEATURES:
    X_train_cat[col] = X_train_cat[col].astype(str)
    X_val_cat[col] = X_val_cat[col].astype(str)
    X_test_cat[col] = X_test_cat[col].astype(str)

In [ ]:
from catboost import CatBoostRegressor
import joblib

MODEL_DIR = Path("artifacts/model")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

cat_model = CatBoostRegressor(
    iterations=1000,
    learning_rate=0.05,
    depth=8,
    random_seed=42,
    early_stopping_rounds=100,
    loss_function="RMSE",
    eval_metric="RMSE",
    verbose=100,
)

cat_model.fit(
    X_train_cat,
    y_train,
    cat_features=CATEGORICAL_FEATURES,
    eval_set=(X_val_cat, y_val),
)

joblib.dump(
    cat_model,
    MODEL_DIR / "catboost.pkl",
)

0:	learn: 35.3199380	test: 32.5534722	best: 32.5534722 (0)	total: 780ms	remaining: 12m 59s
100:	learn: 10.8363452	test: 9.9172487	best: 9.9172487 (100)	total: 41.6s	remaining: 6m 10s
200:	learn: 10.4036159	test: 9.5405437	best: 9.5405437 (200)	total: 1m 20s	remaining: 5m 21s
300:	learn: 10.1523592	test: 9.4349953	best: 9.4349887 (299)	total: 1m 57s	remaining: 4m 31s
400:	learn: 10.0116026	test: 9.3882498	best: 9.3880634 (399)	total: 2m 34s	remaining: 3m 51s
500:	learn: 9.9073033	test: 9.3605240	best: 9.3605240 (500)	total: 3m 11s	remaining: 3m 10s
600:	learn: 9.8286248	test: 9.3401964	best: 9.3401964 (600)	total: 3m 47s	remaining: 2m 31s
700:	learn: 9.7586611	test: 9.3291534	best: 9.3291189 (698)	total: 4m 26s	remaining: 1m 53s
800:	learn: 9.6910040	test: 9.3196233	best: 9.3194819 (799)	total: 5m 6s	remaining: 1m 16s
900:	learn: 9.6204847	test: 9.3103509	best: 9.3103509 (900)	total: 6m 1s	remaining: 39.7s
999:	learn: 9.5626408	test: 9.3069610	best: 9.3069610 (999)	total: 6m 53s	remaini

['artifacts\\model\\catboost.pkl']

# light bgm and xg boost


In [ ]:
from sklearn.preprocessing import OrdinalEncoder
from pathlib import Path


encoder = OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1,
)

X_train_enc = X_train.copy()
X_val_enc = X_val.copy()
X_test_enc = X_test.copy()

X_train_enc[CATEGORICAL_FEATURES] = encoder.fit_transform(
    X_train_enc[CATEGORICAL_FEATURES]
)

X_val_enc[CATEGORICAL_FEATURES] = encoder.transform(
    X_val_enc[CATEGORICAL_FEATURES]
)

X_test_enc[CATEGORICAL_FEATURES] = encoder.transform(
    X_test_enc[CATEGORICAL_FEATURES]
)

joblib.dump(
    encoder,
    MODEL_DIR / "encoder.pkl",
)

In [241]:
print(X_train_enc)
print(y_train)
print(X_val_cat)
print(y_val)

        year  month  product_id  hub_id  category  brand  city  product_name  \
0       2023      4        15.0     7.0       3.0    2.0   0.0         103.0   
1       2023      4       499.0     9.0       5.0    6.0   3.0         327.0   
2       2023      4       499.0     8.0       5.0    6.0   6.0         327.0   
3       2023      4       499.0     7.0       5.0    6.0   0.0         327.0   
4       2023      4       499.0     6.0       5.0    6.0   7.0         327.0   
...      ...    ...         ...     ...       ...    ...   ...           ...   
111995  2025      2       495.0     7.0       5.0    8.0   0.0         440.0   
111996  2025      2        55.0     6.0       2.0    5.0   7.0         248.0   
111997  2025      2       228.0     0.0       1.0    0.0   2.0           9.0   
111998  2025      2       368.0     7.0       1.0    6.0   0.0         301.0   
111999  2025      2       443.0     7.0       1.0    8.0   0.0         406.0   

        opening_stock  sales_last_1  sa

In [ ]:
from lightgbm import LGBMRegressor

lgb_model = LGBMRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=64,
    random_state=42,
)

lgb_model.fit(
    X_train_enc,
    y_train,
    eval_set=[
        (X_val_enc, y_val)
    ],
)

joblib.dump(lgb_model, MODEL_DIR / "lightgbm.pkl") 

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.030201 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1697
[LightGBM] [Info] Number of data points in the train set: 112000, number of used features: 17
[LightGBM] [Info] Start training from score 78.756321


['artifacts\\model\\lightgbm.pkl']

# XG-boost training


In [ ]:
from xgboost import XGBRegressor

xgb_model = XGBRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=8,
    objective="reg:squarederror",
    random_state=42,
)

xgb_model.fit(
   X_train_enc,
    y_train,
    eval_set=[
        (X_val_enc, y_val)
    ],
    verbose=100,
)

joblib.dump(xgb_model, MODEL_DIR / "xgboost.pkl") 

[0]	validation_0-rmse:32.40000
[100]	validation_0-rmse:10.31688
[200]	validation_0-rmse:10.27396
[300]	validation_0-rmse:10.26321
[400]	validation_0-rmse:10.26191
[500]	validation_0-rmse:10.26378
[600]	validation_0-rmse:10.25987
[700]	validation_0-rmse:10.26023
[800]	validation_0-rmse:10.26426
[900]	validation_0-rmse:10.26244
[999]	validation_0-rmse:10.25993


['artifacts\\model\\xgboost.pkl']

In [245]:
import numpy as np
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

def evaluate(model, X, y):

    pred = model.predict(X)

    return {
        "MAE": mean_absolute_error(y, pred),
        "RMSE": np.sqrt(mean_squared_error(y, pred)),
        "R2": r2_score(y, pred),
        "MAPE": np.mean(np.abs((y - pred) / y)) * 100,
    }

In [246]:
results = {}

results["CatBoost"] = {
    "train": evaluate(cat_model, X_train_cat, y_train),
    "validation": evaluate(cat_model, X_val_cat, y_val),
    "test": evaluate(cat_model, X_test_cat, y_test),
}

results["LightGBM"] = {
    "train": evaluate(lgb_model, X_train_enc, y_train),
    "validation": evaluate(lgb_model, X_val_enc, y_val),
    "test": evaluate(lgb_model, X_test_enc, y_test),
}

results["XGBoost"] = {
    "train": evaluate(xgb_model, X_train_enc, y_train),
    "validation": evaluate(xgb_model, X_val_enc, y_val),
    "test": evaluate(xgb_model, X_test_enc, y_test),
}

In [247]:
import pandas as pd

rows = []

for model_name, datasets in results.items():

    for dataset_name, metrics in datasets.items():

        row = {
            "Model": model_name,
            "Dataset": dataset_name,
        }

        row.update(metrics)

        rows.append(row)

results_df = pd.DataFrame(rows)

print(results_df)

      Model     Dataset       MAE       RMSE        R2       MAPE
0  CatBoost       train  7.659702   9.630158  0.931716  10.096387
1  CatBoost  validation  7.361007   9.306961  0.921707  10.467871
2  CatBoost        test  8.853899  11.154660  0.920480  10.472059
3  LightGBM       train  7.240566   9.184011  0.937896   9.988809
4  LightGBM  validation  7.922934  10.255346  0.904938  11.330979
5  LightGBM        test  9.618417  12.406499  0.901631  11.393184
6   XGBoost       train  6.003804   7.735717  0.955939   8.498208
7   XGBoost  validation  7.923394  10.259927  0.904853  11.293555
8   XGBoost        test  9.585843  12.382924  0.902004  11.354887


In [248]:
models = {
    "CatBoost": cat_model,
    "LightGBM": lgb_model,
    "XGBoost": xgb_model,
}

best_model_name = min(
    results,
    key=lambda x: results[x]["validation"]["RMSE"],
)

best_model = models[best_model_name]

joblib.dump(
    best_model,
    MODEL_DIR / "best_model.pkl",
)

print("Best Model:", best_model_name)

Best Model: CatBoost


In [249]:
best_model = joblib.load(
    MODEL_DIR / "best_model.pkl"
)

encoder = joblib.load(
    MODEL_DIR / "encoder.pkl"
)

# Prediction


In [257]:
from pydantic import BaseModel, Field

class InventoryPredictionRequest(BaseModel):
    year: int = Field(..., ge=2024)
    month: int = Field(..., ge=1, le=12)

    product_id: str
    hub_id: str

    category: str
    brand: str
    city: str
    product_name: str

    opening_stock: float = Field(..., ge=0)

    sales_last_1: float = Field(..., ge=0)
    sales_last_2: float = Field(..., ge=0)
    sales_last_3: float = Field(..., ge=0)

    avg_sales_3: float = Field(..., ge=0)

    promotion: int = Field(..., ge=0, le=1)

    discount_percentage: float = Field(..., ge=0, le=100)

    festival_flag: int = Field(..., ge=0, le=1)

    returns: float = Field(..., ge=0)

In [261]:
from pydantic import BaseModel

class InventoryPredictionResponse(BaseModel):

    product_id: str
    hub_id: str
    predicted_next_month_demand: float
    model_used: str
    product_name:str 
    city: str
    brand: str

In [262]:
import pandas as pd

def predict_inventory(data: InventoryPredictionRequest):

    df = pd.DataFrame([data.model_dump()])

    if best_model_name == "CatBoost":

        for col in CATEGORICAL_FEATURES:
            df[col] = df[col].astype(str)

    else:

        df[CATEGORICAL_FEATURES] = encoder.transform(
            df[CATEGORICAL_FEATURES]
        )

    prediction = float(best_model.predict(df)[0])

    return InventoryPredictionResponse(
        product_id=data.product_id,
        hub_id=data.hub_id,
        predicted_next_month_demand=round(prediction, 2),
        model_used=best_model_name,
        product_name=data.product_name,
        city=data.city,
        brand=data.brand

    )

In [263]:
sample = InventoryPredictionRequest(
    year=2026,
    month=8,
    product_id="P001",
    hub_id="H001",
    category="Electronics",
    brand="Samsung",
    city="Chennai",
    product_name="Samsung M35",
    opening_stock=120,
    sales_last_1=95,
    sales_last_2=100,
    sales_last_3=98,
    avg_sales_3=97.6,
    promotion=1,
    discount_percentage=15,
    festival_flag=0,
    returns=2,
)

result = predict_inventory(sample)

print(result.model_dump()) 

{'product_id': 'P001', 'hub_id': 'H001', 'predicted_next_month_demand': 113.7, 'model_used': 'CatBoost', 'product_name': 'Samsung M35', 'city': 'Chennai', 'brand': 'Samsung'}
